# Inference Explorer

Interactive walkthrough of the linear-probing classification pipeline.

- **Full test set**: evaluate the trained probe on every cached test embedding for an (encoder, task) pair — metrics + a confusion matrix.
- **Single image**: run one real test image through the live frozen encoder + trained probe, see the image, its embedding, the prediction, and the inference time for each stage.

Run `extract_features.py` and `train_probe.py` first — this notebook only reads what they produced (`features/`, `probes/checkpoints/`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from sklearn.metrics import confusion_matrix, multilabel_confusion_matrix

from probes.inference import (
    load_config,
    available_encoders_for_task,
    get_test_dataset,
    run_single_image,
    run_test_set,
)

cfg = load_config()
SINGLE_FRAME_TASKS = cfg["tasks"]["single_frame"]

## Plotting helpers

In [ ]:
def plot_embedding(embedding: np.ndarray, title: str):
    """Heatmap of the raw embedding vector, reshaped close to square, plus basic stats."""
    d = embedding.shape[0]
    side = int(np.ceil(np.sqrt(d)))
    padded = np.full(side * side, np.nan)
    padded[:d] = embedding
    grid = padded.reshape(side, side)

    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(grid, cmap="coolwarm")
    ax.set_title(f"{title}\ndim={d}  norm={np.linalg.norm(embedding):.2f}  mean={embedding.mean():.3f}  std={embedding.std():.3f}")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


def plot_probs(probs: np.ndarray, class_names: list[str], true_label, multi_label: bool):
    fig, ax = plt.subplots(figsize=(6, 3))
    colors = ["tab:green" if c in (true_label if multi_label else [true_label]) else "tab:blue" for c in class_names]
    ax.bar(class_names, probs, color=colors)
    ax.set_ylabel("sigmoid prob" if multi_label else "softmax prob")
    ax.set_ylim(0, 1)
    ax.set_title("Prediction (green = true label)")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def plot_confusion(y_true, y_pred, class_names, multi_label):
    if not multi_label:
        cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
        fig, ax = plt.subplots(figsize=(5, 5))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=30, ha="right")
        ax.set_yticklabels(class_names)
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
        ax.set_title("Confusion matrix")
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()
    else:
        # One small true/false confusion matrix per class (multilabel_confusion_matrix).
        cms = multilabel_confusion_matrix(y_true, y_pred)
        n = len(class_names)
        cols = min(4, n)
        rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
        axes = np.atleast_1d(axes).flatten()
        for i, name in enumerate(class_names):
            ax = axes[i]
            ax.imshow(cms[i], cmap="Blues")
            ax.set_title(name, fontsize=10)
            ax.set_xticks([0, 1]); ax.set_xticklabels(["neg", "pos"])
            ax.set_yticks([0, 1]); ax.set_yticklabels(["neg", "pos"])
            for r in range(2):
                for c in range(2):
                    ax.text(c, r, cms[i, r, c], ha="center", va="center")
        for ax in axes[n:]:
            ax.axis("off")
        fig.suptitle("Per-class confusion matrices (multi-label)")
        plt.tight_layout()
        plt.show()

## Controls

In [ ]:
task_dd = widgets.Dropdown(options=SINGLE_FRAME_TASKS, description="Task:")
encoder_dd = widgets.Dropdown(options=available_encoders_for_task(task_dd.value, cfg), description="Encoder:")
mode_radio = widgets.RadioButtons(options=["Full test set", "Single image"], description="Mode:")
qid_dd = widgets.Dropdown(options=list(get_test_dataset(task_dd.value, cfg)["qID"]), description="qID:")
random_btn = widgets.Button(description="\U0001F3B2 Random image")
run_btn = widgets.Button(description="Run", button_style="primary")
single_image_box = widgets.HBox([qid_dd, random_btn])
out = widgets.Output()


def _on_task_change(change):
    encoder_dd.options = available_encoders_for_task(task_dd.value, cfg)
    qid_dd.options = list(get_test_dataset(task_dd.value, cfg)["qID"])


def _on_mode_change(change):
    single_image_box.layout.display = "" if mode_radio.value == "Single image" else "none"


def _on_random_click(_):
    qid_dd.value = np.random.choice(qid_dd.options)


task_dd.observe(_on_task_change, names="value")
mode_radio.observe(_on_mode_change, names="value")
random_btn.on_click(_on_random_click)
_on_mode_change(None)

display(widgets.VBox([task_dd, encoder_dd, mode_radio, single_image_box, run_btn, out]))

## Run

In [ ]:
def run(_):
    out.clear_output()
    with out:
        encoder, task = encoder_dd.value, task_dd.value
        if mode_radio.value == "Full test set":
            result = run_test_set(encoder, task, cfg)
            print(f"{encoder} / {task}  (n={len(result.y_true)})")
            for k, v in result.metrics.items():
                print(f"  {k}: {v:.4f}")
            plot_confusion(result.y_true, result.y_pred, result.class_names, result.multi_label)
        else:
            qid = qid_dd.value
            result = run_single_image(encoder, task, qid, cfg)

            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(result.image)
            ax.set_title(f"qID={qid}")
            ax.axis("off")
            plt.show()

            plot_embedding(result.embedding, title=f"{encoder} embedding")
            plot_probs(result.probs, result.class_names, result.true_label, result.multi_label)

            print(f"true label:      {result.true_label}")
            print(f"predicted label: {result.pred_label}")
            print()
            print("--- inference time (image fed in -> classification ready) ---")
            print(f"  1. preprocess + encoder forward: {result.encode_ms:8.2f} ms")
            print(f"  2. linear probe forward:         {result.probe_ms:8.2f} ms")
            print(f"  =  END-TO-END TOTAL:             {result.total_ms:8.2f} ms")


run_btn.on_click(run)
